In [ ]:
import os
import glob
import pandas as pd

"""
Fuse all **hourly resampled** Parquet tables (those ending in ``_resampled.parquet``)
into a single wide dataset **and then attach the static demographics** as
encounter‑level columns.

Output → ``/home/lkapral/hb/data/resampled/hourly_fused.parquet``

### Key points
* Every resampled file is outer‑joined on ``(encounterId, utcChartTime)`` – the
  timestamp grid is already aligned to 1‑hour intervals.
* Demographics (age / sex_or_gender) are **NOT** resampled; we read the original
  parquet, drop duplicates per encounter, and join once at the very end.
* Duplicate data columns created by outer merges are coalesced (first
  non‑null wins), so you don’t get ``_x`` / ``_y`` suffix clutter.
"""

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RESAMPLED_PATH = "/home/lkapral/hb/data/resampled/"
DEMOGRAPHIC_FILE = "/home/lkapral/hb/data/CIS.parquet"
OUT_FILE = os.path.join(RESAMPLED_PATH, "hourly_fused.parquet")

# If you want to fuse only certain resampled files, list their basenames here;
# otherwise the script picks up *all* ``*_resampled.parquet`` in the folder.
FILES_TO_FUSE = None  # e.g. ["blood-pressure-…_resampled.parquet", …]

# Down‑cast toggle (saves ~memory)
DOWNSHIFT = True

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def downcast_df(df: pd.DataFrame) -> pd.DataFrame:
    if not DOWNSHIFT:
        return df
    out = df.copy()
    float_cols = out.select_dtypes(include="float").columns.difference(["encounterId"])
    int_cols = out.select_dtypes(include="int").columns.difference(["encounterId"])
    for c in float_cols:
        out[c] = pd.to_numeric(out[c], downcast="float")
    for c in int_cols:
        out[c] = pd.to_numeric(out[c], downcast="integer")
    return out


def merge_coalesce(left: pd.DataFrame, right: pd.DataFrame) -> pd.DataFrame:
    m = left.merge(
        right,
        on=["encounterId", "utcChartTime"],
        how="outer",
        suffixes=("", "__dup"),
        copy=False,
    )
    dup_cols = [c for c in m.columns if c.endswith("__dup")]
    for dup in dup_cols:
        base = dup[:-5]
        if base in m.columns:
            m[base] = m[base].combine_first(m[dup])
            m.drop(columns=dup, inplace=True)
        else:
            m.rename(columns={dup: base}, inplace=True)
    return m

# ---------------------------------------------------------------------------
# Step 1 – discover hourly resampled files
# ---------------------------------------------------------------------------
pattern = os.path.join(RESAMPLED_PATH, "*_resampled.parquet")
all_files = sorted(glob.glob(pattern))

# Exclude demographic if it was accidentally resampled (not expected)
all_files = [p for p in all_files if not os.path.basename(p).startswith("demographic-")]

files = (
    [os.path.join(RESAMPLED_PATH, f) for f in FILES_TO_FUSE]
    if FILES_TO_FUSE is not None
    else all_files
)
if not files:
    raise FileNotFoundError(f"No resampled Parquet files found in {RESAMPLED_PATH}.")

print(f"Fusing {len(files)} hourly tables …")

# ---------------------------------------------------------------------------
# Step 2 – iterative outer merge on (encounterId, utcChartTime)
# ---------------------------------------------------------------------------
base: pd.DataFrame | None = None
for i, path in enumerate(files, start=1):
    fname = os.path.basename(path)
    print(f"  [{i}/{len(files)}] {fname}")
    df = pd.read_parquet(path)
    df["encounterId"] = df["encounterId"].astype("int32")
    df["utcChartTime"] = pd.to_datetime(df["utcChartTime"])
    print(f"{fname} shape:", df.shape)
    base = df if base is None else merge_coalesce(base, df)

print("Merged hourly shape:", base.shape)

# ---------------------------------------------------------------------------
# Step 3 – attach demographics once per encounter
# ---------------------------------------------------------------------------
print("Attaching demographics from", os.path.basename(DEMOGRAPHIC_FILE))

demo_cols = ["encounterId", "age", "sex_or_gender"]  # adjust list as needed
if not os.path.exists(DEMOGRAPHIC_FILE):
    raise FileNotFoundError(f"Demographic parquet not found: {DEMOGRAPHIC_FILE}")

demo = pd.read_parquet(DEMOGRAPHIC_FILE, columns=demo_cols)

# Drop duplicate entries per encounter, keeping the first (they should all match)

demo = demo.drop_duplicates(subset=["encounterId"]).astype({"encounterId": "int32"})
print("  unique encounters:", len(demo))

base = base.merge(demo, on="encounterId", how="left")

# ---------------------------------------------------------------------------
# Step 4 – final tidy up & write
# ---------------------------------------------------------------------------
base = downcast_df(base)
base.sort_values(["encounterId", "utcChartTime"], inplace=True)

print("Final fused shape:", base.shape)
print("Writing →", OUT_FILE)
base.to_parquet(OUT_FILE)
print("Fusion complete 🚀")


In [ ]:
import os
import glob
import pandas as pd

"""
Fuse all **hourly resampled** Parquet tables (those ending in ``_resampled.parquet``)
into a single wide dataset **and then attach the static demographics** as
encounter‑level columns.

Output → ``/home/lkapral/hb/data/resampled/hourly_fused.parquet``

### Key points
* Every resampled file is outer‑joined on ``(encounterId, utcChartTime)`` – the
  timestamp grid is already aligned to 1‑hour intervals.
* Demographics (age / sex_or_gender) are **NOT** resampled; we read the original
  parquet, drop duplicates per encounter, and join once at the very end.
* Duplicate data columns created by outer merges are coalesced (first
  non‑null wins), so you don’t get ``_x`` / ``_y`` suffix clutter.
"""

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RESAMPLED_PATH = "/home/lkapral/hb/data/resampled/"

OUT_FILE = os.path.join(RESAMPLED_PATH, "hourly_fused.parquet")

base =  pd.read_parquet(os.path.join(RESAMPLED_PATH, "hourly_fused.parquet"))

In [ ]:
base['encounterId'].nunique()

In [ ]:
merged['patientId'].nunique()